# Beni wake-word training (Colab)

End-to-end notebook that builds a microWakeWord model for the
wake phrase **"Beni"** (and its natural variants like "Hey Beni").

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**. CPU also works but is
   ~5-8× slower for training.
2. Run all cells. The final cell downloads `hey_beni.tflite` to your
   browser; drop it into `firmware/main/wake_word/model/`.

The helper scripts live in `ml/wake_word/scripts/` of the repo — this
notebook clones the repo into Colab and imports them, so every edit
to those scripts survives across training runs via git.

Total runtime: ~30-45 min end to end on a T4.

## 1. Install dependencies

Pins chosen to match microWakeWord's current public release. TF is
pre-installed in Colab; we install on top rather than replacing to
avoid restart-the-runtime dances.

In [ ]:
!pip install -q \
    'piper-tts>=1.2.0' \
    'microwakeword' \
    'datasets>=2.18.0' \
    'soundfile' \
    'librosa>=0.10'

# Sanity: confirm GPU is actually attached. If this prints nothing,
# go back to Runtime → Change runtime type. The training cell will
# still work on CPU but you'll be waiting a while.
!nvidia-smi -L || echo '(no GPU — training will run on CPU)'

## 2. Clone the repo for helper scripts

We could inline these into the notebook, but keeping them as
committed `.py` files means every training run is pinned to a
specific commit of the data-gen logic — so if a model regresses we
can diff the synthesis script against the last good run.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/Ammer-Tech/heybeni.git'
REPO_DIR = Path('/content/heybeni')

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

sys.path.insert(0, str(REPO_DIR / 'ml' / 'wake_word'))

from scripts.synthesize_positives import synthesize_positives, write_manifest
from scripts.fetch_negatives import fetch_common_voice_negatives
from scripts.export_model import (
    export_to_tflite,
    write_metrics_report,
    dump_as_c_header,
)

WW_ROOT = Path('/content/wake_word')
WW_ROOT.mkdir(exist_ok=True)
ARTIFACTS = WW_ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

## 3. Synthesize positives

Downloads ~18 Piper voices (once, ~600 MB total), then renders
~4000 "Beni" / "Hey Beni" clips across prosody variants. The voice
list and prosody grid are defined in `synthesize_positives.py` — edit
there, not inline, so runs stay reproducible.

In [ ]:
plan = synthesize_positives(
    out_dir=WW_ROOT / 'data' / 'positive',
    voices_dir=WW_ROOT / 'voices',
    seed=42,
    samples_per_combo=3,
)
write_manifest(plan, ARTIFACTS / 'positives_manifest.json')
print('positive clips:', len(plan.produced_files))

## 4. Fetch negatives (Common Voice English)

Streams ~5000 short English clips from Common Voice. Needs no auth —
CV is open — but the HF downloader sometimes asks for a token in
Colab. If you see an auth prompt, `huggingface-cli login` in a fresh
cell with a read-only token will fix it.

In [ ]:
num_neg = fetch_common_voice_negatives(
    out_dir=WW_ROOT / 'data' / 'negative',
    num_samples=5000,
    seed=42,
)
print('negative clips:', num_neg)

## 5. Build microWakeWord feature tensors

microWakeWord's training expects spectrograms, not raw WAVs. The
`microwakeword` package ships a `FeatureExtractor` that turns a
directory of WAVs into (T, F) log-Mel tensors matching the same
front-end the firmware will run at inference time. Using their
extractor — not a hand-rolled librosa pipe — is critical: a mismatch
between training features and firmware features is the #1 way a
wake word works at training time and never fires on device.

In [ ]:
from microwakeword.data.audio_utils import generate_features_for_clip
from microwakeword.data.clips import ClipsHandler
import numpy as np

# microWakeWord's standard front-end: 40-band log-Mel, 10 ms hop,
# 25 ms window, 1.5 s clip length. The firmware replicates these
# exact numbers — any divergence here must be mirrored there.
FEATURE_CONFIG = dict(
    window_size_ms=25,
    window_stride_ms=10,
    desired_samples=16000 * 1.5,  # 1.5 s @ 16 kHz
    sample_rate=16000,
    num_mel_bins=40,
)

pos_handler = ClipsHandler(
    input_directory=str(WW_ROOT / 'data' / 'positive'),
    label=1,
    feature_config=FEATURE_CONFIG,
)
neg_handler = ClipsHandler(
    input_directory=str(WW_ROOT / 'data' / 'negative'),
    label=0,
    feature_config=FEATURE_CONFIG,
)

# Returns list[(spec, label)]. Materialise into numpy arrays here so
# the training cell sees plain tensors, not a streaming iterator —
# microWakeWord's trainer wants a Keras-style in-memory fit call.
pos_features = pos_handler.load_features()
neg_features = neg_handler.load_features()

X = np.stack([f for f, _ in pos_features + neg_features])
y = np.array([lbl for _, lbl in pos_features + neg_features])
print('feature tensor:', X.shape, 'labels:', y.shape, 'positives:', int(y.sum()))

## 6. Train MixedNet

microWakeWord's default MixedNet is a small convolutional model
(~20-40 kB int8) that's been tuned specifically for ESP32-class
microcontrollers. We use its packaged trainer with mild overrides
for epoch count and validation split. 30 epochs is roughly where
MixedNet plateaus on a 5k positive / 5k negative split — pushing to
60+ overfits the Piper voices badly.

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from microwakeword.models.mixednet import MixedNet

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

# Add channel dim if the feature extractor returned (T, F).
if X_train.ndim == 3:
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]

model = MixedNet.build(
    input_shape=X_train.shape[1:],
    num_classes=2,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    # Halve LR when val loss plateaus — standard wake-word recipe;
    # without this MixedNet tends to oscillate around ~2% val loss.
    callbacks=[
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=6, restore_best_weights=True,
        ),
    ],
)

## 7. Evaluate and pick a threshold

The int8 output of MixedNet is a logit; the firmware compares it
against a fixed threshold and fires if wake-class score > threshold.
We log FRR (missed wake) at a handful of FAR (false firing) points
and pick the threshold the firmware will ship with.

In [ ]:
import numpy as np

probs = model.predict(X_val, verbose=0)[:, 1]  # P(wake)

metrics = {'operating_points': []}
for target_far in (0.001, 0.005, 0.01, 0.02):
    # Find the threshold that yields target_far on negatives.
    neg_scores = probs[y_val == 0]
    threshold = float(np.quantile(neg_scores, 1.0 - target_far))
    pos_scores = probs[y_val == 1]
    frr = float((pos_scores < threshold).mean())
    metrics['operating_points'].append({
        'target_far': target_far,
        'threshold': threshold,
        'frr_at_threshold': frr,
    })
    print(f'FAR={target_far:.3f} -> threshold={threshold:.3f}, FRR={frr:.3f}')

# Default firmware threshold: pick the FAR=0.005 operating point
# unless that leaves us with awful FRR; then ease off to 0.01.
default = min(
    metrics['operating_points'],
    key=lambda op: op['frr_at_threshold'] + 10 * max(0, op['target_far'] - 0.01),
)
metrics['default_threshold'] = default['threshold']
metrics['default_frr'] = default['frr_at_threshold']
metrics['default_far'] = default['target_far']
print('chosen threshold:', default)

## 8. Export int8 TFLite + C header

Quantise with a subset of training spectrograms, write both a raw
.tflite (for re-quantisation / debugging) and a C header that the
firmware includes directly.

In [ ]:
import random as _random

# Use 300 random training spectrograms for calibration — enough to
# cover the activation distribution without slowing quantisation to
# a crawl.
_random.seed(42)
cal_idx = _random.sample(range(len(X_train)), min(300, len(X_train)))
cal_samples = [X_train[i] for i in cal_idx]

tflite_path = ARTIFACTS / 'hey_beni.tflite'
header_path = ARTIFACTS / 'hey_beni_model.h'

size_bytes = export_to_tflite(model, cal_samples, tflite_path)
dump_as_c_header(tflite_path, header_path, var_name='hey_beni_tflite')
write_metrics_report(metrics, ARTIFACTS / 'metrics.json')

print(f'.tflite size: {size_bytes/1024:.1f} kB')
print('artifacts:')
!ls -la {ARTIFACTS}

## 9. Download to your machine

Triggers three browser downloads. Drop `hey_beni.tflite` and
`hey_beni_model.h` into `firmware/main/wake_word/model/`; commit
`metrics.json` next to them so we have a record of which model
shipped.

In [ ]:
from google.colab import files

files.download(str(ARTIFACTS / 'hey_beni.tflite'))
files.download(str(ARTIFACTS / 'hey_beni_model.h'))
files.download(str(ARTIFACTS / 'metrics.json'))